In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
db_path = project_root / "data" / "cloudtrail.db"

connection = sqlite3.connect(db_path)

In [2]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_events FROM events",
    connection
)

,total_events
0,95


In [3]:
pd.read_sql_query(
    """
    SELECT
        user,
        COUNT(*) as event_count
    FROM events
    GROUP BY user
    ORDER BY event_count DESC
    """,
    connection
)

,user,event_count
0,admin,36
1,developer,22
2,service-user,19
3,analyst,18


In [4]:
# Create a lookup using pandas

users = pd.read_sql_query(
    "SELECT DISTINCT user FROM events",
    connection
)

users

,user
0,admin
1,developer
2,analyst
3,service-user


In [5]:
print("Number of rows in users:", len(users))
print("How many unique values: ", users["user"].nunique())

assert len(users) == users["user"].nunique()
assert users["user"].isnull().sum() == 0

Number of rows in users: 4
How many unique values:  4


In [6]:
users["department"] = "Unassigned"
users["role"] = "Unknown"

In [7]:
# Adding fictional data
users.loc[users["user"] == "admin", "department"] = "IT"
users.loc[users["user"] == "admin", "role"] = "Administrator"

users.loc[users["user"] == "developer", "department"] = "Engineering"
users.loc[users["user"] == "developer", "role"] = "Data Engineer"

users.loc[users["user"] == "analyst", "department"] = "Finance"
users.loc[users["user"] == "analyst", "role"] = "Data Analyst"

users.loc[users["user"] == "service-user", "department"] = "IT"
users.loc[users["user"] == "service-user", "role"] = "Service Account"

users.head()

,user,department,role
0,admin,IT,Administrator
1,developer,Engineering,Data Engineer
2,analyst,Finance,Data Analyst
3,service-user,IT,Service Account


In [8]:
users.to_sql(
    name="identities",
    con=connection,
    if_exists="replace",
    index=False
)

4

In [9]:
pd.read_sql_query(
    "SELECT * FROM identities",
    connection

)

,user,department,role
0,admin,IT,Administrator
1,developer,Engineering,Data Engineer
2,analyst,Finance,Data Analyst
3,service-user,IT,Service Account


In [10]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS identity_rows,
        COUNT(DISTINCT user) AS unique_users
    FROM identities
    """,
    connection
)

,identity_rows,unique_users
0,4,4


In [11]:
df_joined = pd.read_sql_query(
    """
    SELECT 
        e.timestamp,
        e.user,
        e.action,
        i.department
    FROM events as e
    LEFT JOIN identities as i
        ON e.user=i.user""",
        connection
)

In [12]:
assert len(df_joined) == 95

print(df_joined["department"].isnull().sum())

0


In [13]:
users_duplicate = users.copy()

In [14]:
data = [{'user': "admin", 'department': "Security", "role": "Security Administrator"}]
false_row = pd.DataFrame(data, columns=["user", "department", "role"])

In [15]:
users_duplicate = pd.concat([users_duplicate, false_row], ignore_index=True)
users_duplicate.head(5)

,user,department,role
0,admin,IT,Administrator
1,developer,Engineering,Data Engineer
2,analyst,Finance,Data Analyst
3,service-user,IT,Service Account
4,admin,Security,Security Administrator


In [16]:
print("Total identity rows:", len(users_duplicate))
print("Unique identities:", users_duplicate["user"].nunique())

Total identity rows: 5
Unique identities: 4


In [17]:
users_duplicate.to_sql(
    name="identities_duplicate",
    con=connection,
    if_exists="replace",
    index=False
)

5

In [18]:
admin_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS admin_events
    FROM events
    WHERE user = 'admin'
    """,
    connection
)

admin_count

,admin_events
0,36


In [19]:
df_joined_duplicate = pd.read_sql_query(
    """
    SELECT
        e.timestamp,
        e.user,
        e.action,
        i.department
    FROM events AS e
    LEFT JOIN identities_duplicate AS i
        ON e.user = i.user
    """,
    connection
)

In [20]:
print("Original events:", 95)
print("Joined rows:", len(df_joined_duplicate))
print("Extra rows:", len(df_joined_duplicate) - 95)

Original events: 95
Joined rows: 131
Extra rows: 36


In [21]:
admin_events = admin_count["admin_events"].iloc[0]

assert len(df_joined_duplicate) == 95 + admin_events

In [22]:
pd.read_sql_query(
    """
SELECT
    user,
    COUNT(*) AS identity_count
FROM identities_duplicate
GROUP BY user
HAVING COUNT(*) = 2""",
connection
)

,user,identity_count
0,admin,2


In [23]:
duplicate_count = users_duplicate["user"].duplicated().sum()

print("Duplicate identity keys:", duplicate_count)

assert users["user"].duplicated().sum() == 0

Duplicate identity keys: 1


In [24]:
df_joined_clean = pd.read_sql_query(
    """
    SELECT
        e.timestamp,
        e.user,
        e.action,
        i.department
    FROM events AS e
    LEFT JOIN identities AS i
        ON e.user = i.user
    """,
    connection
)

assert len(df_joined_clean) == 95

### Day 18 — SQL Joins and Join Cardinality

Created an identity lookup table containing unique users and fictional department and role metadata. Joined the lookup to our SQLite CloudTrail events table using a `LEFT JOIN`.

**Key accomplishments:**
- Practiced `LEFT JOIN` and `INNER JOIN` concepts.
- Created and validated an identity lookup with unique, non-null user keys.
- Successfully joined identity metadata to 95 CloudTrail events without increasing the row count.
- Intentionally introduced a duplicate `admin` identity to investigate join cardinality.
- Observed the joined dataset increase from 95 to 131 rows, creating 36 additional rows (37.9% inflation).
- Used `GROUP BY`, `HAVING`, and pandas assertions to detect duplicate identity keys.

**Key takeaway:** SQL joins can inflate record counts when the lookup table contains duplicate keys. Validating key uniqueness and comparing row counts before and after joins helps prevent inaccurate aggregations and misleading analytical results.

**Limitation:** The identity metadata is fictional, and the duplicate record was intentionally introduced for learning purposes. In a production environment, conflicting identity attributes would require investigation rather than arbitrary deduplication.